# Credit Card Fraud Detection using XGBoost and SMOTE

This notebook demonstrates how to detect rare fraudulent transactions.

Workflow:
1. Create an imbalanced dataset.
2. Split training and testing data.
3. Apply SMOTE only to training data.
4. Train XGBoost.
5. Predict fraud probabilities.
6. Tune the decision threshold.
7. Inspect feature importance.

This uses synthetic data for learning; a real project would use a cleaned transaction dataset.


In [ ]:
# Install these packages in a separate cell if needed:
# %pip install numpy pandas matplotlib scikit-learn imbalanced-learn xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


## 1. Create a synthetic transaction dataset

Fraud is usually rare. The `weights` argument makes fraudulent transactions a small minority.


In [ ]:
X, y = make_classification(
    n_samples=10000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    weights=[0.98, 0.02],  # Approximately 2% fraud
    random_state=42
)

feature_names = [
    "transaction_amount", "transaction_time_score", "device_risk",
    "location_risk", "recent_transaction_count", "account_age_score",
    "payment_risk", "merchant_risk"
]

df = pd.DataFrame(X, columns=feature_names)
df["is_fraud"] = y

display(df.head())
print(df["is_fraud"].value_counts())
print("Fraud percentage:", round(df["is_fraud"].mean() * 100, 2), "%")


## 2. Split the data

`X` contains transaction features. `y` contains the target:
- `0` = legitimate
- `1` = fraudulent

Stratification keeps a similar fraud percentage in both sets.


In [ ]:
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training fraud rate:", round(y_train.mean() * 100, 2), "%")
print("Testing fraud rate:", round(y_test.mean() * 100, 2), "%")


## 3. Apply SMOTE

**SMOTE** creates synthetic minority-class examples instead of simply copying existing fraud rows.

It is applied only to the training set so the test set remains realistic and untouched.


In [ ]:
print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train, y_train
)

print("\nAfter SMOTE:")
print(pd.Series(y_train_resampled).value_counts())


## 4. Train XGBoost

**XGBoost** builds decision trees sequentially. Each new tree tries to correct errors made by earlier trees, allowing it to learn complex nonlinear patterns.


In [ ]:
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train_resampled, y_train_resampled)
print("Training completed.")


## 5. Predict fraud probabilities

`predict_proba()` returns class probabilities. Column `1` is the estimated probability that a transaction is fraudulent.


In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "actual_fraud": y_test.values,
    "fraud_probability": y_prob
})

display(results.head(10))


## 6. Evaluate ranking performance

- **ROC-AUC** measures how well the model ranks fraud above legitimate transactions.
- **PR-AUC** is particularly useful for imbalanced data because it focuses on precision and recall.


In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print(f"ROC-AUC: {roc_auc:.3f}")
print(f"PR-AUC:  {pr_auc:.3f}")


## 7. Tune the decision threshold

The default threshold is usually `0.5`, but fraud detection may use a lower threshold because missing fraud can be costly.

A lower threshold usually increases recall, but it can also increase false positives.


In [ ]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
threshold_results = []

for threshold in thresholds:
    # Convert probabilities into 0/1 predictions.
    y_pred_threshold = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        "recall": recall_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        "f1_score": f1_score(
            y_test, y_pred_threshold, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(3))


## 8. Evaluate a selected threshold

We use `0.30` only as a demonstration. In a real system, select the threshold using validation data and the costs of false positives and false negatives.


In [ ]:
chosen_threshold = 0.30
y_pred = (y_prob >= chosen_threshold).astype(int)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


## 9. Inspect feature importance

Feature importance shows which features the trained XGBoost model relied on when building its trees.

High importance does not prove that a feature causes fraud; it only means that the feature helped the model make predictions.


In [ ]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df)

plt.figure(figsize=(9, 5))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.gca().invert_yaxis()
plt.xlabel("Feature importance")
plt.title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()
